# Data Analysis Red Bus

In [1]:
!pip install pandas numpy matplotlib seaborn scikit-learn tensorflow
!pip install xgboost lightgbm catboost optuna -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.9/395.9 kB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 15.4 MB/s eta 0:00:00


In [2]:
#Mounting drive to get my data
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
#Load path where my dataset is
path = "/content/drive/MyDrive/RedBusAnalysis"

In [4]:
#Importing pandas
import pandas as pd

In [5]:
#Loading data set
train_data = pd.read_csv(path + "/train.csv")
transaction_data = pd.read_csv(path + "/transactions.csv")
test_data = pd.read_csv(path + "/test_8gqdJqH.csv")

# Feature Extraction

In [6]:
#Filtering for prediction 15 days before journey
transaction_15 = transaction_data[transaction_data["dbd"] == 15]

In [7]:
#Creating unique route key to match later with test dataset
transaction_15["route_key"] = transaction_15["doj"] + "_" + transaction_15["srcid"].astype(str) + "_" + transaction_15["destid"].astype(str)

<ipython-input-7-1703587420>:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  transaction_15["route_key"] = transaction_15["doj"] + "_" + transaction_15["srcid"].astype(str) + "_" + transaction_15["destid"].astype(str)


In [8]:
#Selecting Relevant feature
features = transaction_15[["route_key", "cumsum_seatcount", "cumsum_searchcount", "srcid_region", "destid_region", "srcid_tier", "destid_tier"]]

In [9]:
#Merge with train labels
train_data["route_key"] = train_data["doj"] + "_" + train_data["srcid"].astype(str) + "_" + train_data["destid"].astype(str)
#Drop if existing feature in train data to avoid collision during merge
cols_to_drop = ["cumsum_seatcount", "cumsum_searchcount",
                "srcid_region", "destid_region", "srcid_tier", "destid_tier"]

train_data = train_data.drop(columns=[col for col in cols_to_drop if col in train_data.columns])

train_data = train_data.merge(features, on="route_key", how="left")
train_data.dropna(inplace=True)

In [10]:
#Mergin with test set
duplicate_cols = [
    "cumsum_seatcount", "cumsum_searchcount",
    "srcid_region", "destid_region",
    "srcid_tier", "destid_tier"
]

# Drop them from test_data if they exist
test_data = test_data.drop(columns=[col for col in duplicate_cols if col in test_data.columns])
test_data = test_data.merge(features, on="route_key", how="left")
test_data['cumsum_seatcount'] = test_data['cumsum_seatcount'].fillna(0)
test_data['cumsum_searchcount'] = test_data['cumsum_searchcount'].fillna(0)
for col in ["srcid_region", "destid_region", "srcid_tier", "destid_tier"]:
    test_data[col] = test_data[col].fillna("Unknown")

In [11]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [12]:
# Encoding categorical features safely
categorical_features = ["srcid_region", "destid_region", "srcid_tier", "destid_tier"]
for col in categorical_features:
    le = LabelEncoder()

    # Combine train + test categories for fitting
    combined_values = pd.concat([train_data[col], test_data[col]], axis=0).astype(str)

    # Fit encoder on all possible values
    le.fit(combined_values)

    # Transform separately
    train_data[col] = le.transform(train_data[col].astype(str))
    test_data[col] = le.transform(test_data[col].astype(str))

In [13]:
#Select final features
features = ["cumsum_seatcount", "cumsum_searchcount"]
target = "final_seatcount"

In [56]:
X_train = train_data[features]
y_train = train_data[target]
X_test = test_data[features]

In [57]:
#Normalizing features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Model Training

In [58]:
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
import optuna

In [59]:
#Split data
# X_train_df, X_test_df, y_train_df, y_test_df = train_test_split(
#     X, y, test_size=0.2, random_state=42
# )

In [60]:
#Model 1 : XGBoost
xgb = XGBRegressor(n_estimators=100, random_state=42)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)

In [61]:
#Model 2: LGBMRegressor
lgb = LGBMRegressor(n_estimators=100, random_state=42)
lgb.fit(X_train, y_train)
lgb_pred = lgb.predict(X_test)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001976 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 67200, number of used features: 2
[LightGBM] [Info] Start training from score 2001.729464


In [62]:
#Model 3: CatBooster
cat = CatBoostRegressor(verbose=0, iterations=100, random_state=42)
cat.fit(X_train, y_train)
cat_pred = cat.predict(X_test)

In [63]:
#Model 4: Deep Learning LSTM/GRU
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler

In [64]:
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [65]:
#Reshape Data for LSTM
X_seq_train = X_train_scaled.reshape((X_train_scaled.shape[0], 1, X_train_scaled.shape[1]))
X_seq_test = X_test_scaled.reshape((X_test_scaled.shape[0], 1, X_test_scaled.shape[1]))

In [66]:
lstm_model = Sequential([
    LSTM(32, activation='relu', input_shape=(1, X.shape[1])),
    Dense(1)
])
lstm_model.compile(optimizer='adam', loss='mse')

In [68]:
lstm_model.fit(X_seq_train, y_train, epochs=10, batch_size=32, verbose=0)

In [69]:
lstm_pred = lstm_model.predict(X_seq_test).flatten()

185/185 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [80]:
#Combining prediction results
ensemble_pred = (xgb_pred + lgb_pred + cat_pred + lstm_pred) / 4

In [82]:
submission = test_data[["route_key"]].copy()
submission["final_seatcount"] = ensemble_pred


In [83]:
submission.to_csv("submission_file.csv", index=False)

In [84]:
#Download submission file
from google.colab import files
files.download("submission_file.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [85]:
submission

,route_key,final_seatcount
0,2025-02-11_46_45,2902.336970
1,2025-01-20_17_23,1836.176263
2,2025-01-08_02_14,890.939674
3,2025-01-08_08_47,890.939674
4,2025-01-08_09_46,890.939674
...,...,...
5895,2025-01-23_46_48,3399.362456
5896,2025-02-21_46_09,890.939674
5897,2025-01-17_32_19,1649.592637
5898,2025-01-24_45_03,890.939674
